# SLIIT IT3051 - Data Mining & Predictive Analytics
## EV Battery Prognostics & Health Management (PHM) Dual-Task System
### Group: Necrons | Phase 1: EDA, Data Cleaning & Preprocessing (Week 1 / Viva 1)

---

### Team Roles & Modular Work Breakdown:
| Person | Topic Owned | Key Deliverables |
| :--- | :--- | :--- |
| **Person A** | **Data Structure, Schema & Missingness Audit** | Schema inspection, variable types, duplicate checks, 67-column missingness breakdown, sensor sanity screening |
| **Person B** | **Distributions, Outliers & Treatment Decisions** | Univariate distributions, RUL bell-curve analysis, IQR/Z-score outlier detection, physical outlier justification |
| **Person C** | **Imbalance, Multicollinearity & Feature Engineering** | Class imbalance (18,616 vs 1,384), correlation heatmaps, bivariate degradation analysis, 4 domain features |
| **Person D** | **Data Leakage Guard, Splits & ColumnTransformer** | Target Isolation, ID exclusion, stratified train/test split, production ColumnTransformer pipeline |


### 0. Environment Setup & Global Imports


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Visual formatting settings
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.sans-serif'] = 'Arial'

print('Libraries loaded successfully!')


---
# Section A: Data Structure, Schema & Missingness Audit
**Owner: Person A**
**Focus:** Dataset ingestion, metadata verification, data types, duplicate records, sensor sanity bounds, and missing value patterns.


In [ ]:
# Load the raw dataset
DATASET_PATH = 'ev battery_failure  Dataset.csv'
df_raw = pd.read_csv(DATASET_PATH)

print('=== DATASET DIMENSIONS ===')
print(f'Total Rows (Records):   {df_raw.shape[0]:,}')
print(f'Total Columns (Fields): {df_raw.shape[1]}')


#### A.1 Data Types & Duplicate Check


In [ ]:
# Check duplicates
duplicates = df_raw.duplicated().sum()
print(f'Duplicate Rows Detected: {duplicates}')

# Identify Categorical vs Numerical Columns
cat_cols_initial = df_raw.select_dtypes(include=['object', 'string']).columns.tolist()
num_cols_initial = df_raw.select_dtypes(include=[np.number]).columns.tolist()

print(f'Categorical Columns ({len(cat_cols_initial)}): {cat_cols_initial}')
print(f'Numerical Columns ({len(num_cols_initial)}): {num_cols_initial[:10]}... (total {len(num_cols_initial)})')


#### A.2 Missing Value Audit (MCAR Analysis)


In [ ]:
# Missing Value Breakdown across all 70 columns
missing_counts = df_raw.isnull().sum()
cols_with_missing = missing_counts[missing_counts > 0].sort_values(ascending=False)
missing_pct = (cols_with_missing / len(df_raw)) * 100

missing_summary = pd.DataFrame({
    'Missing Count': cols_with_missing,
    'Missing Pct (%)': missing_pct.round(2)
})

print(f'Total columns with missing values: {len(missing_summary)} out of {df_raw.shape[1]}')
print(f'Missing percentage ranges strictly between {missing_pct.min():.2f}% and {missing_pct.max():.2f}%')
print('\nTop 10 columns by missing rate:')
print(missing_summary.head(10))

print('\nTarget Missing Values:')
print(f'predicted_remaining_life_cycles (RUL): {df_raw["predicted_remaining_life_cycles"].isnull().sum():,} missing')
print(f'battery_failure (Classification):     {df_raw["battery_failure"].isnull().sum():,} missing')


#### A.3 Physical Boundary & Sensor Sanity Screening


In [ ]:
# Physical Boundary Checks: Impossible Sensor Values
sanity_checks = {
    'Max Cell Temp > 120 C (Extreme Runaway)': (df_raw['cell_temperature_max'] > 120).sum(),
    'Avg Cell Temp < -40 C (Deep Freeze)':      (df_raw['cell_temperature_avg'] < -40).sum(),
    'Cell Voltage Avg <= 0 V':                 (df_raw['cell_voltage_avg'] <= 0).sum(),
    'Pack Voltage <= 0 V':                     (df_raw['pack_voltage'] <= 0).sum(),
    'Internal Resistance <= 0 Ohm':            (df_raw['internal_resistance'] <= 0).sum()
}

print('=== SENSOR SANITY SCREENING RESULTS ===')
for check, count in sanity_checks.items():
    print(f'  {check}: {count} violations')


**Person A Viva Notes:**
- The dataset contains 20,000 records and 70 features with 0 duplicate rows.
- Exactly 67 features exhibit missing values between 3.05% and 4.94% (Missing Completely at Random - MCAR).
- Continuous features will be imputed with their training median (resilient to sensor fluctuations); vehicle attributes with mode.
- The Task 1 target (`predicted_remaining_life_cycles`) contains 898 missing entries; these are filtered out for regression training to avoid synthetic label leakage.
